# OSEM cho D710 — lắp bốn số hạng hiệu chỉnh của GE vào `sirf.STIR`

$$y = S\,(G x) + b$$

| ký hiệu | nội dung | file | sinh ra bởi | vào OSEM qua |
|---|---|---|---|---|
| `y` | prompt thô | `<ca>/decoded/bed<n>.hs` | `d710 decode` | `recon.set_input` |
| `b` | randoms + scatter | `<ca>/work/bed<n>/background.hs` | `d710 estimate` → `d710 tostir` | `set_background_term` |
| `S` | norm × dead time × suy giảm | `<ca>/work/bed<n>/normdt.hs` × af | ↑ + `utils/attenuation.py` | `set_acquisition_sensitivity` **trước** `set_up` |
| `G` | — | — | STIR | `AcquisitionModelUsingRayTracingMatrix` |

Cả bốn số hạng hiệu chỉnh — randoms, scatter, normalisation, dead time — **lấy
bằng chính kernel của GE** (`pet_recon` chạy dưới gdb trong container), không tự
dựng lại bằng Python. Suy giảm là số hạng duy nhất dựng ở đây, từ CT.

## Notebook này KHÔNG chứa mã

Mọi `def` sống trong `utils/` (không thuộc thuật toán) hoặc `osem/` (thuật
toán); ở đây chỉ còn import, đặt tham số, gọi hàm và vẽ hình.
`tests/test_notebook_contract.py` **fail** nếu một cell định nghĩa `def`/`class`
hay dài quá 15 câu lệnh.

Đó không phải câu nệ hình thức: dự án này đã một lần chép `utils/` vào notebook
rồi hai bản lệch nhau, và cả hai vẫn chạy — chỉ là không còn tính cùng một thứ.


## 0. Dựng đầu vào — một lệnh cho CẢ CA

`d710 exam` chạy đủ ba bước cho **mọi bed** trong thư mục raw. Ca nhi có 6 bed
(`SINO0000..0005` → bed 1..6), mỗi bed vài phút.

```bash
export D710_OUT=~/UET/d710_out             # đầu ra đi đâu; KHÔNG có mặc định
conda activate petct_reconstruction        # SIRF/STIR chỉ nạp khi env đã activate

D710/d710 exam --raw ~/Documents/11082026/petRDFS/NQLHQWPU/SYQGRVWD/QONDOBON \
               --ct  ~/Documents/11082026/PESI/p1/e1/s2 \
               --case ped
```

Bed nào đã xong thì bỏ qua, nên chạy lại sau khi hỏng giữa chừng là an toàn
(`--force` để làm lại). Ba bước nó gọi, nếu muốn chạy tay từng cái:

```bash
d710 decode   --raw <petRDFS/.../DIR> --case ped            # 1. RDF -> Interfile
d710 estimate --raw <petRDFS/.../DIR> --ct <CT> --case ped  # 2. kernel GE -> .f32
d710 tostir   --case ped                                    # 3. -> Interfile STIR
```

Cả ba chạy **trong container**; bước 1 và 3 chỉ cần `docker`, bước 2 thêm
`python3` trần. Notebook (OSEM + xuất ảnh) là phần duy nhất cần môi trường
project, vì SIRF không có trong image.

Ra ở đâu:

```
$D710_OUT/ped/
    decoded/        bed<n>.hs/.s/.json      <- y
    vendor/bed<n>/  *.f32 của GE
    work/bed<n>/    randoms/scatter/background/normdt/norm_only/attn .hs/.s
    export/         ped_bqml.nii.gz, ped_suvbw.nii.gz, dicom/
    scratch/        tmp_*.hs của SIRF — xoá lúc nào cũng được
```


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from osem import recon, stitch
from utils import attn, export, plots, quant, sirf_env, terms
from utils.geometry import PLANE_MM
from utils.paths import case

CASE = "ped"                       # <- ca đang chạy
C = case(CASE)                     # đọc $D710_OUT; thiếu nó là LỖI, không đoán
BEDS = C.beds()                    # dò TỪ ĐĨA: chỉ bed đã xong cả ba bước
HDRS = {n: C.header(n) for n in BEDS}

sirf_env.setup(C)                  # chdir vào scratch + chặn INFO của STIR
terms.bed_table(C, BEDS)

## 1. Suy giảm — số hạng duy nhất KHÔNG lấy từ vendor

`d710 estimate` có dùng mu-map (nó phải có, để mô phỏng scatter), nhưng cái nó
xuất ra là **scatter**, không phải hệ số suy giảm. Nên `af` dựng lại ở đây từ
cùng series CT, bằng `utils/attenuation.py`.

Series CT lấy từ sidecar của chính bed, không gõ tay: suy giảm và scatter phải
dựng từ **cùng một** CT, và đó là CT mà kernel của GE đã thấy.

Ràng buộc chống ghép nhầm ca: `FrameOfReferenceUID` của CT phải **bằng đúng**
`sop_instance_uid` trong header RDF của bed. Đây là đồng nhất thức, không phải
phép so gần đúng — thư mục ảnh nằm cạnh thư mục raw **không** đảm bảo cùng exam
(`11082026/` chứa ảnh của hai exam khác nhau). `utils.attn` kiểm mỗi bed.

`af` cache vào `work/bed<n>/attn.hs`, nên sau lần chạy đầu cả bốn số hạng của
mô hình đều có mặt trên đĩa. ⚠ Xoá `attn.hs`/`attn.s` nếu đổi CT hoặc đổi lưới ảnh.


In [ ]:
CT = terms.ct_dir(C, BEDS[0])              # CT mà d710 estimate đã dùng
y0, x0 = recon.image_grid(C, BEDS[0])      # lưới chung mọi bed: 47 × 328 × 328
VOX = [float(v) for v in x0.voxel_sizes()]  # (z, y, x) mm

AT = attn.Attenuation(C, CT, x0, y0)
print(AT.describe())
print(f"ảnh {x0.as_array().shape}  "
      f"voxel {VOX[2]:.4f} × {VOX[1]:.4f} × {VOX[0]:.4f} mm\n")
print("suy giảm theo bed (nước ở 511 keV ≈ 0.096 1/cm):")
AF = AT.all(BEDS)

In [ ]:
# Một lượt qua TẤT CẢ các bed: nạp -> rút lát cắt -> trả RAM ngay.
# Đỉnh bộ nhớ là MỘT bed (~2,5 GB) chứ không phải sáu (~15 GB).
PROJ, STATS, P0 = terms.collect(C, BEDS, AF)

## 2. Toàn bộ sinogram

Mỗi số hạng vẽ hai lát cắt của cùng khối `(1, 553, 288, 381)`:

* **hàng trên** — sinogram ngang `view × tangential` tại **một plane trực tiếp**
  (segment 0). Đây là "sinogram" theo nghĩa thường gặp.
* **hàng dưới** — **gộp theo view**, còn `plane × tangential`, đủ cả 553 plane.
  Vạch đứt trắng là ranh giới cuối segment 0 (plane 47): phía dưới nó là các
  segment xiên `±1, ±2, …`, mỗi segment ngắn dần `47 − 4|k|` plane. Nhìn hàng
  này là thấy ngay cách xếp michelogram — và thấy ngay nếu ánh xạ plane sai.

Hai nhóm để riêng vì **đơn vị khác nhau**: miền count (count/bin) và hệ số
(không thứ nguyên). Gộp chung một thang màu thì cả hai đều vô nghĩa.


In [ ]:
for n in BEDS:
    sub = f"{CASE} bed {n} (table {HDRS[n]['table_position_mm']:.0f} mm)"
    plots.sinogram_grid({t: PROJ[n, t] for t in terms.COUNT_TERMS}, P0[n],
                        terms.COUNT_TERMS, "Miền count", "inferno",
                        "count/bin", sub)
    plots.sinogram_grid({t: PROJ[n, t] for t in terms.FACTOR_TERMS}, P0[n],
                        terms.FACTOR_TERMS, "Hệ số (độ nhạy)", "viridis",
                        "không thứ nguyên", sub)
plt.show()

In [ ]:
# Ảnh 2D cho thấy hình dạng; muốn so ĐỘ LỚN thì phải xem lát cắt 1D.
plots.profiles(PROJ, P0, BEDS, CASE)
plots.per_plane(PROJ, BEDS)
plt.show()

## 3. Tái tạo từng bed, rồi ghép trục

Mỗi bed tái tạo độc lập trên lưới 47 × 328 × 328 của riêng nó, sau đó ghép theo
`z = table_position + i·PLANE_MM` — **đúng công thức `attenuation.mu_image` dùng
để cắt CT cho bed đó**, nên hình học hai bên nhất quán theo cấu tạo chứ không
phải nhờ trùng hợp.

### Vùng chồng là chỗ CẢ HAI bed yếu nhất — nên phải đánh trọng số

Bước bàn 124,26 mm = **đúng 38 plane**, bed dài 47 plane → chồng 9 plane. Nhưng
9 plane đó là plane 38–46 của bed dưới và plane 0–8 của bed trên — **hai đầu yếu
nhất gặp nhau**. Trọng số là `SENS[n]`, sensitivity image của chính STIR: mẫu số
OSEM chia vào mỗi vòng lặp, đã gồm norm, dead time, suy giảm và cách projector
lấy mẫu LOR thật, và là ảnh **3D** nên trọng số theo TỪNG VOXEL. Với ML Poisson
`Var(x̂) ≈ x / sens`, nên trọng số nghịch đảo phương sai chính là `w ∝ sens`.

### Phân rã phải làm TRƯỚC khi ghép

Sáu bed chụp cách nhau 91 s, bed 6 muộn hơn bed 1 tới 458 s. Ghép thẳng là dán
sáu thời điểm khác nhau vào một khối → gradient giả dọc trục. `stitch` quy hết
về **thời điểm tiêm**, đúng chuẩn `DecayCorrection = START` của DICOM, và dùng
hoạt độ **trung bình trong khung** chứ không phải tức thời lúc bed bắt đầu.

`N_SUB` phải là ước của 288 (số view).


In [ ]:
N_SUB, N_IT = 12, 3        # 288 view: subset phải là ước của 288

# ~90 s/bed trên máy này, sáu bed ~9 phút. SENS[n] là sensitivity image của
# bed n — dùng làm trọng số ghép ở cell sau.
IMG, SENS = recon.reconstruct_all(C, BEDS, AF, x0, n_sub=N_SUB, n_it=N_IT)

In [ ]:
VOL, Z0, DECAY = stitch.stitch(C, BEDS, IMG, SENS)

# KIỂM chỗ ghép bằng SỐ, đừng chỉ nhìn: "tương quan" so hai bản tái tạo ĐỘC LẬP
# của *cùng một đoạn cơ thể*. Đặt sai chiều trục hay lệch một bed thì sụp ngay.
stitch.overlap_report(C, BEDS, IMG, DECAY)

In [ ]:
plots.beds(IMG, HDRS, f"OSEM {N_IT}×{N_SUB} theo bed — {CASE}  [count/voxel]")
plots.whole_body(VOL, VOX, PLANE_MM,
                 f"Toàn thân {len(BEDS)} bed đã ghép — {CASE}  [count/voxel]")
plt.show()

## Kiểm tra bắt buộc trước khi tin con số

Ảnh ra là **count/voxel**, chưa phải Bq/mL. Đổi thang cần hằng số `K`, và `K`
chỉ đúng cho **đúng chuỗi đã đo ra nó** và **đúng một bước voxel** — projector
tích luỹ theo bước voxel chứ không theo thể tích, nên hằng số đo ở 2,1306 mm
dùng lại ở 1,3672 mm đọc cao 1,56×.

Bất biến gộp **theo plane**, không theo bin: sinogram thô ~0,06 count/bin nên
`p < r` đúng ở ~82 % số bin chỉ vì nhiễu Poisson, và so từng bin là vô nghĩa.
`tests/test_pipeline_data.py` chốt đúng những bất biến này ở chỗ *fail* được.


In [ ]:
BAD = terms.invariant_table(C, BEDS, PROJ, STATS)

## 4. Hiệu chuẩn `K` và xuất ảnh

Ảnh tới đây là **count/voxel đã quy về thời điểm tiêm**. Đổi sang Bq/mL cần một
số vô hướng duy nhất:

$$\text{Bq/mL} \;=\; K \cdot x_{\text{count/voxel}}$$

### `K` của GE có trên đĩa

Exam tự khai file WCC ở header (`wcc_cal_uid`), và file đó có thật trong
`/usr/PET/systemConfig/cal/` **bên trong image** — `d710 estimate` đọc nó lúc
chạy và ghi vào `estimate.json`, nên ở đây chỉ cần đọc sidecar:

```
(0019,1006) 3D WCC 20 October 2023
(0019,1007) "Discovery TOF - PET well counter"
(0019,100B) 4.062768936157227      <- hrActivityFactor của chính máy này
```

Job XR của hãng đặt `IgJobReq.hrActivityFactor = 4.113164` (máy selftest của
GE), lệch 1,2 % — dùng số của chính exam, không dùng số của GE.

⚠ Thừa số `1e4` (`quant.WCC_UNIT_SCALE`) là **SUY ĐOÁN** về quy ước đơn vị của
GE, chưa dẫn ra được. Nó cho 68 % liều nằm trong FOV, hợp lý với scan phủ 77 cm
của bệnh nhi cao 118 cm (thiếu chân). Đo trên NEMA mới chốt được. Đặt `K = None`
để lấy mốc theo liều tiêm thay vì WCC.


In [ ]:
K = quant.k_from_wcc(quant.wcc_activity_factor(C, BEDS[0]))
K_DOSE = quant.k_from_dose(VOL, VOX, quant.dose_bq(HDRS[BEDS[0]]))
print(f"K chặn trên (100 % liều trong FOV) = {K_DOSE:,.1f}\n")

R = quant.report(VOL, K if K is not None else K_DOSE, HDRS[BEDS[0]], VOX)

C.export.mkdir(parents=True, exist_ok=True)
nii = export.write_nifti(R["bqml"], str(C.export / f"{CASE}_bqml.nii.gz"),
                         VOX[2], VOX[1], VOX[0], Z0)
dcm = export.write_dicom(R["bqml"], str(C.export / "dicom"), HDRS[BEDS[0]],
                         VOX[2], VOX[1], VOX[0], Z0,
                         series_desc=f"OSEM SIRF {N_IT}x{N_SUB} BQML")
print(f"\nghi {nii}\nghi {len(dcm)} file DICOM -> {C.export / 'dicom'}")
print("   Units=BQML + liều + cân nặng + DecayCorrection=START -> viewer tự "
      "tính SUV;\n   FrameOfReferenceUID = của exam -> tự chồng khít CT.")

In [ ]:
# SUV TUYẾN TÍNH với K, mà K đang là suy đoán: mọi số dưới đây sai đúng bằng
# tỉ lệ K sai. Chưa dùng để kết luận lâm sàng được.
SUVS = quant.suv_table(R["bqml"], R["mask"], HDRS[BEDS[0]])

plots.suv(SUVS["SUVbw"], R["mask"], VOX, PLANE_MM, R["K"], CASE)
plt.show()

# DICOM cố ý KHÔNG xuất SUV: viewer tự tính từ Units=BQML + liều + cân nặng, và
# người đọc đổi được bw/lbm/bsa. Ghim SUV vào DICOM là khoá cứng một lựa chọn.
print("ghi", export.write_nifti(SUVS["SUVbw"],
                                str(C.export / f"{CASE}_suvbw.nii.gz"),
                                VOX[2], VOX[1], VOX[0], Z0))

## Còn thiếu gì

| thành phần | trạng thái |
|---|---|
| randoms | **xong** — kernel GE (từ singles); ΣR/totalDelays ≈ 0,99 mọi bed |
| scatter (SSS) | **xong** — kernel GE, model-based, tail fit |
| normalisation | **xong** — norm 3D của chính máy, tự tra từ header exam |
| dead time | **xong** — `normdt/norm_only`; phụ thuộc tốc độ đếm |
| suy giảm CT | **xong** — `utils/attenuation.py`, khớp frame of reference |
| `.f32` → STIR Interfile | **xong** — `vendor/to_stir.py`, bit-exact mỗi lần chạy |
| span-2 (segment 0 gộp 2 cặp ring) | **xong** — `normdt` gánh sẵn; ĐỪNG nhân thêm |
| hiệu chỉnh phân rã + ghép trục | **xong** — quy về thời điểm tiêm, trọng số = sensitivity image |
| xuất DICOM + NIfTI | **xong** — `utils/export.py`, `Units = BQML` |
| **hằng số `K`** | **CHƯA** |

**`K` là việc còn lại duy nhất.** Không có WCC nào được áp trong `vendor/`, nên
thang tuyệt đối phải tự đo trên NEMA: chạy `d710 exam --case nema`, đặt
`CASE = "nema"` ở cell 2, đo trên vùng nền có nồng độ biết trước.

Cùng một pipeline không cần notebook:

```bash
d710 osem   --case ped
d710 export --case ped --format both
```
